# Band-Pass Butterworth Filter for EEG

Applies a 1-40 Hz band-pass filter to the P4 channel in a single step.

**Dataset**: PhysioNet Auditory EEG (Abo Alzahab et al., 2021)  
**Channels**: P4, Cz, F8, T7  
**Sampling rate**: 1000 Hz

## 1. Install dependencies

In [ ]:
!pip install scipy numpy pandas matplotlib

## 2. Clone the resources repo

In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')

## 3. Download the local EEG dataset

In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.csv')):
    !python data/download_local.py

## 4. Apply band-pass filter (1-40 Hz)

In [ ]:
from scipy import signal
import numpy as np
import matplotlib.pyplot as plt
from utils.eeg_loader import load_local_eeg

# Load local EEG data
timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=1
)
channel_data = eeg_data[:, 0]  # P4 channel
fs = 1000  # Sampling rate (Hz)

# Band-pass Butterworth filter: 1-40 Hz in one step
def butter_bandpass_filter(data, lowcut, highcut, fs, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = signal.butter(order, [low, high], btype='band', analog=False)
    return signal.filtfilt(b, a, data)

filtered_bp = butter_bandpass_filter(
    channel_data, lowcut=1.0, highcut=40.0, fs=fs
)

# Plot raw vs band-pass filtered
n_plot = 5000
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(timestamps[:n_plot], channel_data[:n_plot], label='Raw', color='gray')
axes[0].set_ylabel('EEG (uV)')
axes[0].set_title('Raw EEG (P4)')
axes[1].plot(timestamps[:n_plot], filtered_bp[:n_plot], label='Band-pass', color='blue')
axes[1].set_ylabel('EEG (uV)')
axes[1].set_xlabel('Time (ms)')
axes[1].set_title('Band-pass filtered (1-40 Hz)')
plt.tight_layout()
plt.savefig('bandpass_result.png', dpi=150)
plt.show()